In [1]:
%load_ext sql
%sql sqlite:///characters.db

In [2]:
%%sql

CREATE TABLE IF NOT EXISTS species (
    id          INTEGER PRIMARY KEY AUTOINCREMENT,
    species      TEXT    NOT NULL UNIQUE,
    description TEXT    NOT NULL
);

CREATE TABLE IF NOT EXISTS species_buffs (
    species_id      INTEGER NOT NULL REFERENCES species(id),
    attribute       TEXT    NOT NULL REFERENCES attributes(attribute),
    level_modifier  INTEGER NOT NULL,  
    PRIMARY KEY (species_id, attribute)
);


 * sqlite:///characters.db
Done.
Done.


[]

In [3]:
import sqlite3

SPECIES = [
    ('Human', 'Balanced baseline species with no innate buffs or debuffs.'),
    ('Demon', 'Infernal beings whose raw power far exceeds their mortal counterparts.'),
    ('Angel', 'Celestial warriors with heightened resilience and intellect.'),
    ('Elf',   'Ancient race with unmatched speed and battlefield awareness.'),
    ('Dwarf', 'Stout warriors of immense durability, but slower on their feet.'),
]

BUFFS = [
    ('Demon', 'Strength',   2),
    ('Demon', 'Durability', 1),
    ('Demon', 'Speed',     -1),
    ('Angel', 'Durability', 2),
    ('Angel', 'IQ',         1),
    ('Elf',   'Speed',      1),
    ('Elf',   'BIQ',        1),
    ('Elf',   'Strength',  -1),
    ('Dwarf', 'Durability', 3),
    ('Dwarf', 'Stamina',    1),
    ('Dwarf', 'Speed',     -2),
]


conn = sqlite3.connect('characters.db')
cur  = conn.cursor()

for name, description in SPECIES:
    cur.execute(
        "INSERT OR IGNORE INTO species (species, description) VALUES (?, ?)",
        (name, description)
    )

for species_name, attribute, modifier in BUFFS:
    cur.execute(
        """INSERT OR IGNORE INTO species_buffs (species_id, attribute, level_modifier)
           VALUES ((SELECT id FROM species WHERE species = ?), ?, ?)""",
        (species_name, attribute, modifier)
    )

conn.commit()

# ── Verify ────────────────────────────────────────────────────────────────────

rows = cur.execute("""
    SELECT s.id, s.species, s.description, sb.attribute, sb.level_modifier
    FROM   species s
    LEFT JOIN species_buffs sb ON sb.species_id = s.id
    ORDER  BY s.id
""").fetchall()

for row in rows:
    print(row)

conn.close()


(1, 'Human', 'Balanced baseline species with no innate buffs or debuffs.', None, None)
(2, 'Demon', 'Infernal beings whose raw power far exceeds their mortal counterparts.', 'Durability', 1)
(2, 'Demon', 'Infernal beings whose raw power far exceeds their mortal counterparts.', 'Speed', -1)
(2, 'Demon', 'Infernal beings whose raw power far exceeds their mortal counterparts.', 'Strength', 2)
(3, 'Angel', 'Celestial warriors with heightened resilience and intellect.', 'Durability', 2)
(3, 'Angel', 'Celestial warriors with heightened resilience and intellect.', 'IQ', 1)
(4, 'Elf', 'Ancient race with unmatched speed and battlefield awareness.', 'BIQ', 1)
(4, 'Elf', 'Ancient race with unmatched speed and battlefield awareness.', 'Speed', 1)
(4, 'Elf', 'Ancient race with unmatched speed and battlefield awareness.', 'Strength', -1)
(5, 'Dwarf', 'Stout warriors of immense durability, but slower on their feet.', 'Durability', 3)
(5, 'Dwarf', 'Stout warriors of immense durability, but slower on t